# 02 — Visualizing LCZ Maps and Quantifying Class Areas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/general/02_visualization_area_stats.en.ipynb)

**Learning objective**: learn to turn a raw LCZ classification raster into two things planners and reviewers actually need — a static, publication-ready map (plus an opt-in interactive, zoomable version anyone can explore in a browser), and a quantitative breakdown of how much area each LCZ class occupies. By the end of this notebook you will be able to render an LCZ map with `lcz_plot_map`, switch between discrete-class and continuous-raster visualization modes, and compute/chart per-class areas with `lcz_cal_area` using five different chart styles.

> **Plotting defaults (whole series):** every LCZ4py function that produces a figure returns a **static matplotlib `Figure`** by default; pass `renderer="plotly"` for the interactive Plotly version.

## Why interactive visualization matters for LCZ science communication

The Local Climate Zone scheme (Stewart & Oke, 2012) was designed as a *standardized*, worldwide-comparable classification of urban and natural landscapes into 17 classes (1–10 built types, from compact high-rise to large low-rise; A–G / 11–17 natural land covers, from dense trees to water). A classification is only useful, though, if the people who need to act on it — urban planners, city councils, public-health officers, climate-adaptation teams — can actually read it. A static PNG buried in a PDF report rarely survives the trip from the researcher's desktop to the planning meeting; an interactive map that a stakeholder can pan, zoom, and hover over to see the exact class at a given street corner does.

`lcz_plot_map` (LCZ4py, following the visualization philosophy of the LCZ4r R package, Anjos et al. 2025, *Scientific Reports*, https://www.nature.com/articles/s41598-025-92000-0) renders LCZ rasters as **static matplotlib figures by default** — reproducible PNG/PDF/SVG output you can drop straight into a report or manuscript — and, with `renderer='plotly'`, as Plotly WebGL figures: self-contained HTML that opens in any browser, supports hover tooltips and zoom/pan, and (via the additional `renderer='maplibre'` option) can overlay the classification on a real OpenStreetMap basemap for immediate geographic context.

**Discrete vs. continuous rasters.** Two fundamentally different kinds of raster show up in LCZ workflows, and `lcz_plot_map` handles both:

- **Discrete-class choropleth**: the LCZ map itself — every pixel holds an integer class code (1–17). The right way to render this is a categorical color map with one flat color per class and a legend listing class names, *not* a continuous color gradient (a gradient would visually imply that class 5 is 'between' classes 4 and 6, which is meaningless — the LCZ codes are a nominal, not ordinal, scale).
- **Continuous raster overlay**: derived physical fields on the same grid, e.g. per-pixel Land Surface Temperature (LST) from `lcz_get_lst`, or a morphological parameter from `lcz_get_parameters`. These *are* ordinal/interval data, so a diverging or sequential colorscale (e.g. `RdBu_r`, blue=cold to red=hot) with a colorbar is the correct rendering, and averaging or interpolating between pixel values is meaningful.

`lcz_plot_map`'s `data_type` parameter (`'auto'`, `'lcz'`, `'continuous'`) picks the right mode automatically by inspecting the raster's dtype, but you can force it explicitly. Mixing the two up — e.g. applying a categorical palette to a temperature raster — is a common and avoidable source of misleading urban-climate figures.

**Why area quantification comes right after visualization.** A map answers *where*; a bar/pie/donut/sunburst/treemap of per-class area answers *how much*. Before any deeper urban-climate analysis (UHI intensity, thermal comfort modeling, morphological parameter extraction), it is standard practice to first ask: what fraction of this city is compact high-rise (LCZ 1) vs. open low-rise (LCZ 6) vs. dense trees (LCZ A/11) vs. water (LCZ G/17)? `lcz_cal_area` computes exact per-class area in km² (accounting for the raster's actual pixel size and CRS) and renders it as one of five chart types (a static matplotlib figure by default, interactive with `renderer='plotly'`), each suited to a different communication need.

## Install LCZ4py

In [1]:
#pip install LCZ4py

## Get a real LCZ map to work with

We use Juiz de Fora (Liechtenstein) — a small city whose bounding box keeps the download fast — so this notebook runs end-to-end in a couple of minutes. Everything below applies unchanged to any city.

In [2]:
from LCZ4py import lcz_get_map

map_path = lcz_get_map(city="Juiz de Fora")
print(map_path)

22:07:17 - LCZ4py._internal._lcz_map_engine - INFO - Loading clipped map from local cache.


/Users/co2map/.lcz4r_cache/clipped_7d9d37301b2894f6.tif


## `lcz_plot_map` — discrete-class map (static by default)

Key parameters:
- `x`: path to the LCZ GeoTIFF (or an already-open rasterio dataset).
- `show_legend`: shows the 17-class legend (recommended, on by default).
- `inclusive`: swaps the standard LCZ palette for a colorblind-friendly one (see below).
- `use_webgl` / `max_pixels`: keep large rasters responsive in the Plotly renderer by rendering via WebGL and down-sampling above a pixel threshold (ignored by the static renderer).
- `renderer`: `'static'` (default — a native matplotlib `Figure`, saved as PNG/PDF/SVG), `'plotly'` (interactive WebGL figure), or `'maplibre'` (interactive HTML page with an OpenStreetMap basemap underneath — good for giving stakeholders geographic bearings).
- `data_type`: `'auto'` (default) detects LCZ vs. continuous data from the raster dtype.

It returns an `LCZPlotResult` with `.fig` (a matplotlib `Figure` by default, a Plotly figure with `renderer='plotly'`/`'maplibre'`) and, for `maplibre`, `.html`. In a notebook, display it with `display(result.fig)` — or `result.fig.show()` when you asked for the Plotly renderer.

In [ ]:
from IPython.display import display

from LCZ4py import lcz_plot_map

result = lcz_plot_map(map_path, show_legend=True, title="LCZ map — Juiz de Fora", style="nature")
display(result.fig)

**Reading this map**: each flat-colored patch is one LCZ class — warm reds/oranges for the built types (compact/open low- or mid-rise are typical for a small alpine town like Juiz de Fora), greens for vegetated classes (dense/scattered trees on the surrounding slopes), and blue for water/river. With the opt-in `renderer="plotly"` you can also hover over any pixel to read its exact class from the tooltip. For a town this size, expect the built footprint to be a small fraction of the total mapped area, with natural classes (forest, low plants, bare rock) dominating — a pattern very different from a dense metropolitan LCZ map.

## The colorblind-friendly palette (`inclusive=True`)

The LCZ scheme's standard palette (Stewart & Oke 2012) uses reds/oranges for built classes and greens/blues for natural ones — a scheme that is hard or impossible to distinguish for the ~4-8% of readers with red-green color vision deficiency (the most common form of color blindness). Setting `inclusive=True` swaps in a perceptually distinct, colorblind-safe palette that preserves the same class boundaries and legend. This costs nothing in the code (one keyword) and matters a great deal for the accessibility of published figures, conference posters, and any material shared with a general audience — accessible science communication should be the default, not an afterthought reserved for readers who explicitly ask.

In [ ]:
from IPython.display import display

result_inclusive = lcz_plot_map(map_path, show_legend=True, inclusive=True,
                                title="LCZ map — Juiz de Fora (colorblind-friendly palette)")
display(result_inclusive.fig)

Compare the two figures above: the class boundaries are identical, only the color encoding changed. In any report or paper intended for a broad audience, prefer `inclusive=True`.

## Opt-in: the interactive Plotly renderer

The static figure above is what belongs in a paper, a PDF report, or a slide. When you *do* want the explorable, hoverable map — for a stakeholder workshop or a web page — pass `renderer="plotly"`; every figure-producing function in LCZ4py accepts the same switch.

In [ ]:
result_interactive = lcz_plot_map(
    map_path, show_legend=True, renderer="plotly",
    title="LCZ map — Juiz de Fora (interactive)",
)
result_interactive.fig.show()  # interactive Plotly figure

## Per-class area statistics with `lcz_cal_area`

Key parameters:
- `x`: path to the LCZ GeoTIFF.
- `plot_type`: `'bar'`, `'pie'`, `'donut'`, `'sunburst'`, or `'treemap'`.
- `iplot`: if `False`, skip chart rendering and return only the area `DataFrame`.
- `inclusive`: colorblind-friendly palette, same as `lcz_plot_map`.
- `use_duckdb` / `use_geoarrow`: optional accelerated backends for large rasters (degrade gracefully to Polars-only if not installed).

Returns an `LCZAreaResult` with `.df` (a Polars DataFrame: LCZ class, name, area in km², percent of mapped area), `.fig` (the chart: a matplotlib `Figure` by default, Plotly with `renderer='plotly'`), and `.geoarrow_table` (optional zero-copy polygon export). This is the standard first quantitative step in any LCZ-based urban climate study: before computing UHI intensity or morphological parameters, you want to know what fraction of the city is urban vs. natural land cover.

### Bar chart — best for precise class-by-class comparison

A bar chart is the right default whenever the audience needs to compare exact magnitudes across all 17 classes, or rank classes by area. Bars make small differences between adjacent classes easy to read off, which pie-style charts do not.

In [ ]:
from IPython.display import display

from LCZ4py import lcz_cal_area

area_bar = lcz_cal_area(map_path, plot_type="bar", title="LCZ class area — Juiz de Fora", style="nature", save_extension = "png")
display(area_bar.fig)
area_bar.df

**Reading this**: the tallest bars identify the dominant land cover(s) for the study area. For Juiz de Fora, expect one or two natural classes (forest/dense trees, low plants) to dominate, with each built class contributing a comparatively small share — quantitative confirmation of what the map showed visually above.

### Donut chart — best for a quick 'urban vs. natural' share at a glance

A donut (or pie) chart is preferable when the message is a *proportion* story — e.g. "what fraction of the city is built-up?" — rather than a precise ranking. The visual whole-to-part framing communicates that immediately to a non-technical audience (city council, press release), at the cost of making small-class comparisons harder than a bar chart would.

In [ ]:
from IPython.display import display

area_donut = lcz_cal_area(map_path, plot_type="donut", inclusive=True,
                          title="LCZ class area share — Juiz de Fora")
display(area_donut.fig)

**Reading this**: each slice's angular size is proportional to its area share. Grouping the slices mentally into 'warm-colored' (built, LCZ 1-10) vs. 'cool-colored' (natural, LCZ A-G/11-17) gives an instant read on the urban-vs-natural land-cover balance — exactly the number a planner asks for first.

### Treemap — best for showing hierarchy and relative area with nested detail

A treemap encodes area share as rectangle size (like a donut) but also naturally supports grouping classes into a parent/child hierarchy (e.g. "Built" and "Natural" as parent groups, individual LCZ classes nested inside). It is the preferable choice when you want both the whole-to-part story *and* a compact, information-dense layout that also legibly shows every class's exact label and value in the same figure — useful in a written report where a full page is available but you don't want a wall of bars.

In [ ]:
from IPython.display import display

area_treemap = lcz_cal_area(map_path, plot_type="treemap",
                            title="LCZ class area — Juiz de Fora (treemap)")
display(area_treemap.fig)

**Reading this**: rectangle area is proportional to each class's mapped area; larger rectangles are the dominant classes. Unlike the bar chart, small classes are still visible (as small tiles) rather than compressed against an axis, which is handy when you need every class labeled in one static export for a report figure.

### Getting just the numbers (no chart)

Set `iplot=False` when you only need the area table — e.g. to feed into a downstream statistical analysis or a custom figure elsewhere in a pipeline.

In [8]:
area_df = lcz_cal_area(map_path, iplot=False)
area_df

lcz,count,area_km2,area_perc,lcz_name,lcz_col,lcz_colorblind
i16,u64,f32,f32,str,str,str
2,42,0.39,0.03,"""Compact midrise""","""#D9081C""","""#D8755E"""
3,2852,26.51,1.83,"""Compact lowrise""","""#FF0A22""","""#C98027"""
4,4,0.04,0.0,"""Open highrise""","""#C54F1E""","""#B48C00"""
5,40,0.37,0.03,"""Open midrise""","""#FF6628""","""#989600"""
6,6078,56.509998,3.91,"""Open lowrise""","""#FF985E""","""#739F00"""
…,…,…,…,…,…,…
13,9,0.08,0.01,"""Bush, scrub""","""#628432""","""#6290E5"""
14,10066,93.580002,6.48,"""Low plants""","""#B5DA7F""","""#9E7FE5"""
15,50,0.46,0.03,"""Bare rock or paved""","""#000000""","""#C36FDA"""


In [ ]:
from IPython.display import display

from LCZ4py import lcz_plot_map, lcz_cal_area

map_pub = lcz_plot_map(
    map_path,
    show_legend=True,
    title="LCZ map — publication style",
    style="science",
    isave=True,
    save_extension="png")
display(map_pub.fig)

area_pub = lcz_cal_area(
    map_path,
    plot_type="bar",
    title="LCZ class area — publication style",
    style="nature",
    isave=True,
    save_extension="pdf",
)
display(area_pub.fig)


## Conclusion

You now have two complementary tools for communicating an LCZ classification: `lcz_plot_map` for a publication-ready spatial view (a static matplotlib figure by default, an interactive, explorable Plotly map with `renderer="plotly"`), with a colorblind-friendly option via `inclusive=True` and both a discrete-class mode for LCZ maps and a continuous mode for rasters like LST, and `lcz_cal_area` for precise per-class area quantification across five chart styles (bar for precise comparison, donut/pie for quick proportion stories, sunburst/treemap for hierarchical or information-dense layouts). Together they form the standard first deliverable of any LCZ-based urban climate study: *where* the classes are, and *how much* of the city each one covers.

**Previous**: `01_map_acquisition` (downloading and sourcing LCZ maps).
**Next**: `03_morphological_parameters` — extracting the 34 morphological/thermal parameters (building surface fraction, sky view factor, roughness, etc.) that underlie each LCZ class.